In [1]:
from pathlib import Path
from xlsxwriter import Workbook
import numpy as np
import pandas as pd
import pyodbc
import sys
import os
import time
from datetime import date
import re
import warnings as wr
wr.filterwarnings('ignore')

In [2]:
seasons_as10 = {1:'Kharif', 2:'Phase-I', 3:'Phase-II', 4:'Phase-III', 5:'Summer'}
seasons_as20 = {1:'Kharif', 2:'E.Kharif', 3:'L.Kharif', 4:'Rabi', 5:'Summer'}

In [3]:
def clean_rc123(details_1p0):
    
    #get data from the table - RC1
    year = details_1p0['year']
    state_cd = details_1p0['state_cd']
    season_cd = details_1p0['season_cd']
    season_nm = 'kharif' if season_cd in [1,2,3] else 'rabi' if season_cd in [4,5] else 'Season Incorrect'
    st_cd = details_1p0['st_cd']
    novp = details_1p0['novp']
    novcod = details_1p0['novcod']
    ddg = details_1p0['ddg']
    ddtrs = details_1p0['ddtrs']
    crop_list = details_1p0['crop_list']

    
    #set database file path
    folder_path = Path('./software')
    for file_path in folder_path.iterdir():
        if file_path.is_file():
            if season_nm.lower() in file_path.name.lower():
                accdb_path = "./" + "/".join(str(file_path).split("\\"))
    
    #create connection with mdb database
    conn_str = (
        r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
        f'DBQ={accdb_path};'
    )
    conn = pyodbc.connect(conn_str)
    
    cols1 = ['RC', 'ST', 'YR', 'SESON', 'NOVP', 'NOVCOD', 'STAT', 'DIST', 'STRA',
           'VILL', 'EPC', 'NOVPC', 'NOVTRS', 'CADC', 'TLC', 'MAC', 'MUC', 'DDG',
           'SGC', 'GCC', 'LFOG', 'ROG', 'DDTRS', 'TRSSC', 'TRSSF', 'DCKC', 'HSSN',
           'GEOA', 'TCROPA', 'TNSSN', 'TGEOASN', 'RC61', 'RC62', 'RC63', 'REJ']
    
    sql_query1 = 'SELECT * FROM "Copy Of RC1"'
    df1 = pd.read_sql(sql_query1, conn)[cols1]
    
    df1 = (df1
        .drop_duplicates()
        .apply(pd.to_numeric, errors='coerce')
        .query("STAT == @state_cd and SESON == @season_cd and ST == @st_cd")
        .assign(
            REJ = lambda x : np.where(x['REJ']==1, 1, 0)
            ,NOVPC1 = lambda x : x['NOVPC'].fillna(1)
            ,NOVPC = lambda x : np.where(x['NOVPC1']==0, 1, x['NOVPC1'])
            ,NOVTRS1 = lambda x : x['NOVTRS'].fillna(1)
            ,NOVTRS = lambda x : np.where(x['NOVTRS1']==0, 1, x['NOVTRS1'])
            ,CADC = lambda x : np.where(np.isin(x['CADC'], [1,2,3,4]), x['CADC'], 9)
            ,TLC = lambda x : np.where(x['CADC']==3, 0, np.where(np.isin(x['CADC'], [4,9]), 8, x['TLC']))
            ,MAC = lambda x : np.where(x['MAC']==1, 1, 0)
            ,MUC = lambda x : np.where(x['MAC']==1, x['MUC'], None)
            ,TRSSC = lambda x : np.where(x['TRSSC']==3, 8, x['TRSSC'])
            ,NOVP = novp
            ,DDG = ddg
            ,DDTRS = ddtrs
            ,TCROPA = lambda x : x['TCROPA'].fillna(0)
            ,GEOA = lambda x : x['GEOA'].fillna(0)
            ,YR = year+1
            ,NOVCOD = novcod
            ,LFOG = lambda x : np.where(~np.isin(x['LFOG'], [0,1]), None, x['LFOG'])
            ,ROG = lambda x : np.where(~np.isin(x['ROG'], [0,1]), None, x['ROG'])
            ,TRSSF = lambda x : np.where(x['TRSSF'].isna(), None, x['TRSSF'])
        )
        .drop(columns = ['NOVPC1','NOVTRS1'])
    )
    
    non_rej_rc1 = df1[df1['REJ']!=1]
    rej_rc1 = df1[df1['REJ']==1].assign(
    	EPC = None
    	,NOVPC = None
    	,NOVTRS = None
    	,CADC = None
    	,TLC = None
    	,MAC = None
    	,MUC = None
    	,DDG = None
    	,SGC = None
    	,GCC = None
    	,LFOG = None
    	,ROG = None
    	,DDTRS = None
    	,TRSSC = None
    	,TRSSF = None
    	,DCKC = None
    	,HSSN = None
    	,GEOA = None
    	,TCROPA = None
    	,TNSSN = None
    	,TGEOASN = None
    	,RC61 = None
    	,RC62 = None
    	,RC63 = None
    )
    rc1 = pd.concat([non_rej_rc1, rej_rc1], ignore_index=True)
    
    
    
    
    #get data from the table - RC2
    crop_codes_as1p0 = [key for details['']]
    cols2 = ['RC', 'ST', 'YR', 'SESON', 'STAT', 'DIST', 'STRA', 'VILL', 'EPC', 
             'HSSN', 'TNSSN', 'CROP', 'VARC', 'ARSU', 'ARSI', 'ARPU', 'ARPI']
    
    sql_query2 = 'SELECT * FROM "Copy Of RC2"'
    df2 = pd.read_sql(sql_query2, conn)[cols2]
    df2 = (df2
        .drop_duplicates()
        .apply(pd.to_numeric, errors='coerce')
        .query("STAT == @state_cd and SESON == @season_cd and ST == @st_cd")
        .assign(
            YR = year+1
            ,VARC = lambda x : np.where(x['CROP']==0, 0, x['VARC'])
            ,CROP = lambda x : np.where(x['VARC']==0, 0, x['CROP'])
        )
    )
    rc2 = df2[df2['CROP'].isin(crop_list)]    
    
    
    
    
    
    #get data from the table - RC3
    cols3 = ['RC', 'ST', 'YR', 'SESON', 'STAT', 'DIST', 'STRA',
             'VILL', 'SN', 'CROP', 'VARC', 'IRRC', 'ERC']
    
    sql_query3 = 'SELECT * FROM "Copy Of RC3"'
    df3 = pd.read_sql(sql_query3, conn)[cols3]
    df3 = (df3
        .apply(pd.to_numeric, errors='coerce')
        .query("STAT == @state_cd and SESON == @season_cd and ST == @st_cd")
        .assign(
            YR = year+1
            ,VARC = lambda x : np.where(x['CROP']==0, 0, x['VARC'])
            ,CROP = lambda x : np.where(x['VARC']==0, 0, x['CROP'])
        )
    )
    
    rc1_bkp = rc1.copy()
    #creating schedule unique id in RC1
    rc1['unique_id'] = rc1['STAT'].astype(str) + "_" + rc1['SESON'].astype(str) + "_" + rc1['ST'].astype(str) + "_" + rc1['STRA'].astype(str) + "_" + rc1['VILL'].astype(str)+ "_" + rc1['DIST'].astype(str)
    
    
    #creating schedule unique id in df3
    df3['unique_id'] = df3['STAT'].astype(str) + "_" + df3['SESON'].astype(str) + "_" + df3['ST'].astype(str) + "_" + df3['STRA'].astype(str) + "_" + df3['VILL'].astype(str)+ "_" + df3['DIST'].astype(str)
    
    #Left Join on df3 with rc1_epc1
    rc1_epc = rc1[['unique_id','EPC']]
    rc3 = pd.merge(df3, rc1_epc, how='left', left_on='unique_id', right_on='unique_id')
    rc3 = rc3[rc3['EPC'] == 1].drop(columns=['unique_id','EPC'])
    
    
    
    
    
    conn.close()
    return rc1_bkp, rc2, rc3

In [4]:
def clean_rc7(details_2p0):

    #get data from the table - RC7
    year = details_2p0['year']
    state_cd = details_2p0['state_cd']
    season_cd = details_2p0['season_cd']
    st_cd = details_2p0['st_cd']
    crop_nexp = details_2p0['nexp']
    crop_ccf = details_2p0['ccf']
    
    #set database file path
    folder_path = Path('./software')
    for file_path in folder_path.iterdir():
        if file_path.is_file():
            if 'NSSAS' in file_path.name:
                mdb_path = "./" + "/".join(str(file_path).split("\\"))
    
    #create connection with mdb database
    conn_str = (
        r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
        f'DBQ={mdb_path};'
    )
    conn = pyodbc.connect(conn_str)
    
    #get data from the table - Sch20RC7
    cols7 = ['RC', 'ST', 'YR', 'SESON', 'CROP', 'NEXP', 'NEXNR', 'CONFAC', 
             'STAT', 'DIST', 'VILL', 'RSV', 'EXPNO', 'STAGIN', 'REXML', 
             'CCE', 'CFC', 'RCSSN', 'E1', 'E2', 'E9', 'E10', 'E11', 'E12', 
             'ADH', 'QTYG', 'QTYP', 'E13', 'EQST', 'EQSB', 'EQSW', 'EQSP', 
             'E14', 'STYPE', 'E3', 'SVARC', 'SEEDR', 'E4', 'IRR', 'E5', 
             'IRRP', 'NIRR', 'FERTC', 'E6', 'QTN', 'CODN', 'QTP', 'CODP', 
             'QTK', 'CODK', 'MANUR', 'E7', 'PEST', 'E8', 'E15', 'REJ', 
             'CROPER', 'CROPCON', 'AFFECT']
    
    sql_query7 = 'SELECT * FROM "Sch20RC7"'
    rc7 = pd.read_sql(sql_query7, conn)[cols7]
    rc7 = (rc7
        .drop_duplicates()
        .apply(pd.to_numeric, errors="coerce")
        .query("STAT == @state_cd and SESON == @season_cd")
        .assign(
            #adjusting cotton crop code
            YR = year+1
            ,CROP = lambda x : x['CROP'].replace({55: 12, 57: 56})
            ,REJ = lambda x : x['REJ'].apply(lambda y : 1 if y==1 else 0)
            ,RSV = lambda x: x['RSV'].where(x["RSV"].isin([0, 1, 2, 3, 9]), 0)
            ,REXML = lambda x: x["REXML"].where(
                        ~(x["STAGIN"].isin([2, 3]) & ~x["REXML"].isin([1, 2, 3, 4, 9])),
                        9
            )
            ,CFC = lambda x: x["CFC"].where(
                        ~(x["STAGIN"].isin([1, 2]) & ~x["CFC"].isin([1, 2, 3, 4])),
                        4
            )
            ,RCSSN = lambda x: x["RCSSN"].where(
                ~(x["STAGIN"].isin([1, 2]) & ~x["RCSSN"].isin([0, 1, 2, 3, 9])),
                0
            )
            ,EQST = lambda x: x["EQST"].where(
                ~(x["STAGIN"].isin([1, 2]) & x["EQST"].isna()),
                9
            )
            ,EQSB = lambda x: x["EQSB"].where(
                ~(x["STAGIN"].isin([1, 2]) & x["EQSB"].isna()),
                9
            )
            ,EQSW = lambda x: x["EQSW"].where(
                ~(x["STAGIN"].isin([1, 2]) & x["EQSW"].isna()),
                9
            )
            ,EQSP = lambda x: x["EQSP"].where(
                ~(x["STAGIN"].isin([1, 2]) & x["EQSP"].isna()),
                9
            )
            ,QTYG = lambda x: (
                x["QTYG"]
                .where(x["STAGIN"] != 2, None)
                .where(~(x["STAGIN"].eq(1) & x["CROPCON"].eq(4)), 0)
            )
            ,QTYP = lambda x: (
                x["QTYP"]
                .where(x["STAGIN"] != 2, None)
                .where(~(x["STAGIN"].eq(1) & x["CROPCON"].eq(4)), 0)
            )
            ,E13 = lambda x: x["E13"].where(x["STAGIN"] != 2, None)
            ,AFFECT = lambda x: x['AFFECT'].where(~((x['STAGIN'].isin([1,2])) & (x['CROPCON'].isin([1,2]))), 0)
            ,IRR = lambda x : np.where(x['NIRR']>0, 1, 0)
            ,CODN = lambda x : x['CODN'].where(~(x['QTN']>0), 1)
            ,CODP = lambda x : x['CODP'].where(~(x['QTP']>0), 1)
            ,CODK = lambda x : x['CODK'].where(~(x['QTK']>0), 1)
            ,FERTC = lambda x : np.where(((x['QTN']>0) | (x['QTP']>0) | (x['QTK']>0) | (x['CODN']==9) | (x['CODP']==9) | (x['CODK']==9)), 1, 0)
            
        )
    )
    
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN']==8) & (rc7['CODP']==8) & 
        (rc7['CODK']==8) & ((rc7['QTN']==0) | rc7['QTN'].isna()) & 
        ((rc7['QTP']==0) | rc7['QTP'].isna()) & ((rc7['QTK']==0) | rc7['QTK'].isna())), 'FERTC'] = 0
    
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN']==8) & (rc7['CODP']==8) & 
        (rc7['CODK']==8) & ((rc7['QTN']==0) | rc7['QTN'].isna()) & 
        ((rc7['QTP']==0) | rc7['QTP'].isna()) & ((rc7['QTK']==0) | rc7['QTK'].isna())), 'QTN'] = np.nan
    
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN']==8) & (rc7['CODP']==8) & 
        (rc7['CODK']==8) & ((rc7['QTN']==0) | rc7['QTN'].isna()) & 
        ((rc7['QTP']==0) | rc7['QTP'].isna()) & ((rc7['QTK']==0) | rc7['QTK'].isna())), 'QTP'] = np.nan
    
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN']==8) & (rc7['CODP']==8) & 
        (rc7['CODK']==8) & ((rc7['QTN']==0) | rc7['QTN'].isna()) & 
        ((rc7['QTP']==0) | rc7['QTP'].isna()) & ((rc7['QTK']==0) | rc7['QTK'].isna())), 'QTK'] = np.nan
    
    rc7.loc[~((rc7['CODN'].isin([1,8,9])) | (rc7['CODN'].isna())), 'CODN'] = np.nan
    rc7.loc[~((rc7['CODP'].isin([1,8,9])) | (rc7['CODP'].isna())), 'CODP'] = np.nan
    rc7.loc[~((rc7['CODK'].isin([1,8,9])) | (rc7['CODK'].isna())), 'CODK'] = np.nan
    
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN'].isna()) & (rc7['CODP'].isna()) & (rc7['CODK'].isna())), 'FERTC'] = 0
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN'].isna()) & (rc7['CODP'].isna()) & (rc7['CODK'].isna())), 'QTN'] = np.nan
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN'].isna()) & (rc7['CODP'].isna()) & (rc7['CODK'].isna())), 'QTP'] = np.nan
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN'].isna()) & (rc7['CODP'].isna()) & (rc7['CODK'].isna())), 'QTK'] = np.nan
    
    rc7.loc[(rc7['FERTC']==0), 'QTN'] = np.nan
    rc7.loc[(rc7['FERTC']==0), 'QTP'] = np.nan
    rc7.loc[(rc7['FERTC']==0), 'QTK'] = np.nan
    rc7.loc[(rc7['FERTC']==0), 'CODN'] = 8
    rc7.loc[(rc7['FERTC']==0), 'CODP'] = 8
    rc7.loc[(rc7['FERTC']==0), 'CODK'] = 8
    
    lost_cols = ['CCE', 'CFC', 'RCSSN', 'E1', 'E2', 'E9', 'E10', 'E11', 'E12', 
                 'ADH', 'QTYG', 'QTYP', 'E13', 'EQST', 'EQSB', 'EQSW', 'EQSP',
                 'E14', 'STYPE', 'E3', 'SVARC', 'SEEDR', 'E4', 'IRR', 'E5',
                 'IRRP', 'NIRR', 'FERTC', 'E6', 'QTN', 'CODN', 'QTP', 'CODP',
                 'QTK', 'CODK', 'MANUR', 'E7', 'PEST', 'E8', 'E15','CROPER', 
                 'CROPCON', 'AFFECT']
    
    for col in lost_cols:
        rc7.loc[rc7['STAGIN']==3, col] = np.nan
    
    for crop,plan in crop_nexp.items():
        rc7.loc[((rc7['ST']==st_cd) & (rc7['CROP']==crop)), 'NEXP'] = plan
        received = rc7[(rc7['ST']==st_cd) & (rc7['CROP']==crop)].shape[0]
        rc7.loc[((rc7['ST']==st_cd) & (rc7['CROP']==crop)), 'NEXNR'] = received
        
    rc7['NEXNR'] = rc7['NEXP'].sub(rc7['NEXNR'], fill_value=0)
    
    for crop,ccf in crop_ccf.items():
        if ccf<1:
            ccf = np.floor(ccf*10000)
        rc7.loc[(rc7['CROP']==crop), 'CONFAC'] = ccf
        
    
    conn.close()
    return rc7

In [5]:
if __name__ == "__main__":
    
    #Details from Meta_Data file
    state = pd.read_excel('Meta_Data.xlsx', sheet_name='state')
    st = pd.read_excel('Meta_Data.xlsx', sheet_name='st')
    
    #AS1.0 details from Meta_Data file
    as1p0_season = pd.read_excel('Meta_Data.xlsx', sheet_name='as1p0_season')
    as1p0_sch = pd.read_excel('Meta_Data.xlsx', sheet_name='as1p0_sch')
    as1p0_crops = pd.read_excel('Meta_Data.xlsx', sheet_name='as1p0_crops')

    #AS2.0 details from Meta_Data file
    as2p0_season = pd.read_excel('Meta_Data.xlsx', sheet_name='as2p0_season')
    as2p0_sch = pd.read_excel('Meta_Data.xlsx', sheet_name='as2p0_sch')

    #setting variables
    state_code = int(state.loc[0,'state_cd'])
    year = int(state.loc[0,'year'])
    st_codes = list(st['st_cd'])
    season_codes_as1p0 = list(as1p0_season['season_cd'])
    season_codes_as2p0 = list(as2p0_season['season_cd'])

    #creating blank dataframes
    rc1 = pd.DataFrame()
    rc2 = pd.DataFrame()
    rc3 = pd.DataFrame()
    rc7 = pd.DataFrame()
    
    for season_code in season_codes_as1p0:
        for st_code in st_codes:
            details_1p0 = {
                'year' : year,
                'state_cd' : state_code,
                'season_cd' : season_code,
                'st_cd' : st_code,
                'novp' : int(as1p0_sch[(as1p0_sch['season_cd']==season_code) & (as1p0_sch['st_cd']==st_code)]['novp']),
                'novcod' : int(as1p0_sch[(as1p0_sch['season_cd']==season_code) & (as1p0_sch['st_cd']==st_code)]['novcod']),
                'ddg' : int(as1p0_sch[(as1p0_sch['season_cd']==season_code) & (as1p0_sch['st_cd']==st_code)]['ddg']),
                'ddtrs' : int(as1p0_sch[(as1p0_sch['season_cd']==season_code) & (as1p0_sch['st_cd']==st_code)]['ddtrs']),
                'crop_list' : list(as1p0_crops[as1p0_crops['season_cd']==season_code]['crop_cd'])
            }
            r1, r2, r3 = clean_rc123(details_1p0)

            rc1 = pd.concat([rc1, r1], ignore_index = True)
            rc2 = pd.concat([rc2, r2], ignore_index = True)
            rc3 = pd.concat([rc3, r3], ignore_index = True)



    crop_nexp = details['nexp']
    crop_ccf = details['ccf']
    
    for season_code in season_codes_as2p0:
        for st_code in st_codes:
            details_2p0 = {
                'year' : year,
                'state_cd' : state_code,
                'season_cd' : season_code,
                'st_cd' : st_code,
                'nexp' : 
                'ccf' : 
            }
            r7 = clean_rc7(details_2p0)
            rc7 = pd.concat([rc7, r7], ignore_index = True)
    
    if (rc1['HSSN'].isna().sum() + rc1[rc1['HSSN'] == 0].shape[0]) > 0:
        print('='*50)
        print('In RC1, HSSN is Zero or Blank. Please Correct...')
        print('='*50)
    if (rc1['GEOA'].isna().sum() + rc1[rc1['GEOA'] == 0].shape[0]) > 0:
        print('='*50)
        print('In RC1, GEOA is Zero or Blank. Please Correct...')
        print('='*50)

    rc1.sort_values(['ST', 'SESON', 'DIST', 'STRA', 'VILL'], inplace=True)
    rc2.sort_values(['ST', 'SESON', 'DIST', 'STRA', 'VILL', 'CROP', 'VARC'], inplace=True)
    rc3.sort_values(['ST', 'SESON', 'DIST', 'STRA', 'VILL', 'SN'], inplace=True)
    rc7.sort_values(['ST', 'SESON', 'CROP', 'DIST', 'VILL', 'EXPNO'], inplace=True)
    
    rc1.to_excel(excel_writer='RC1_clean.xlsx', sheet_name='RC1', index=False, engine='openpyxl')
    rc2.to_excel(excel_writer='RC2_clean.xlsx', sheet_name='RC2', index=False, engine='openpyxl')
    rc3.to_excel(excel_writer='RC3_clean.xlsx', sheet_name='RC3', index=False, engine='openpyxl')
    rc7.to_excel(excel_writer='RC7_clean.xlsx', sheet_name='RC7', index=False, engine='openpyxl')
    #Generate Annexure Data
    pass

In RC1, HSSN is Zero or Blank. Please Correct...
In RC1, GEOA is Zero or Blank. Please Correct...


In [90]:
as2p0_sch = pd.read_excel('Meta_Data.xlsx', sheet_name='as2p0_sch')

In [91]:
as2p0_sch

,season_cd,crop_cd,crop_nm,plan_cen,plan_state,ccf
0,1,3,Jowar,40,40,0.243640
1,1,5,Bajra,40,40,0.287200
2,1,6,Maize,40,40,0.314520
3,1,15,Groundnut,40,40,0.295120
4,1,33,Gingelly,40,40,0.200000
5,1,11,Redgram,40,40,0.200000
6,1,12,Cotton,40,40,0.065900
7,1,14,Sugarcane,40,40,0.400000
8,2,91,Paddy_I,40,40,0.249399
9,3,92,Paddy_II,60,60,0.248652


In [92]:
as2p0_sch[as2p0_sch['season_cd']==1]

,season_cd,crop_cd,crop_nm,plan_cen,plan_state,ccf
0,1,3,Jowar,40,40,0.24364
1,1,5,Bajra,40,40,0.28720
2,1,6,Maize,40,40,0.31452
3,1,15,Groundnut,40,40,0.29512
4,1,33,Gingelly,40,40,0.20000
5,1,11,Redgram,40,40,0.20000
6,1,12,Cotton,40,40,0.06590
7,1,14,Sugarcane,40,40,0.40000
